# Modules:

In [7]:
import pandas as pd
import time
from google import genai
from google.genai import types
import json
import os
import sys
from tqdm import tqdm
from dotenv import load_dotenv

# Config:

In [13]:
# --- CONFIGURATION ---
# Load the API key from your specific .env file
load_dotenv("API_keys.env")
API_KEY = os.getenv("GEMINI_API_KEY_KOBE")
if API_KEY: print("API_KEY loaded succesfully")

# Prompt Technique Toggles
EXAMPLES_ENABLED = True    # 3-shot examples
REASONING_ENABLED = False   # Chain-of-thought reasoning
CONFIDENCE_ENABLED = False  # Confidence score
MULTISTAGE = 3              # Choose stage 2 or 3 of the multistage process
ROLE_NR = 1                   # Choose the system prompt (role): default = 1
ROLE = {1: "an expert in ESG investment", 
         2: "an expert in classifying posts", 
         3: "a professor of linguistics",
         4: "an academic expert in ESG (Environmental, Social, and Governance) analysis"}[ROLE_NR]

# --- FILES & MODEL ---
MODEL = "Gem2.5f"
MODEL_ID = {"Gem2.5f": "gemini-2.5-flash", "GPT5": "gpt-5"}[MODEL]
OUTPUT_FILE = f"PoC classification_multi{MULTISTAGE}_{MODEL}_role{ROLE_NR}{"_examples" if EXAMPLES_ENABLED else ""}{"_reasoning" if REASONING_ENABLED else ""}{"_confidence" if CONFIDENCE_ENABLED else ""}.csv"
STAGE_ONE_FILE = OUTPUT_FILE.replace(f"_multi{MULTISTAGE}", "") # Continue with the output of stage 1 that has been created by the other classification script
if MULTISTAGE == 3: STAGE_TWO_FILE = OUTPUT_FILE.replace("_multi3", "_multi2") # Stage 2 file only necessary if in the third multistage phase
CATEGORY_FILE = "classification_categories.txt"
EXAMPLES_FILE = f"classification_examples{"_reasoning" if REASONING_ENABLED else ""}{"_confidence" if CONFIDENCE_ENABLED else ""}.txt"

# Script Parameters
TEMPERATURE = 0                       # Strictly deterministic
MAX_POSTS = 10                        # "None" for full analysis
POST_START = 10                       # Index of the post where analysis should start (0 is first post)
CHECKPOINT_INTERVAL = 5              # Save after x posts
SLEEP_TIME = 14                        # Break (in seconds) between API calls
RETRIES = 3                             # Amount of retries for 500 / 503 errors

# The 10 official LSEG subcategories
SUBCATEGORIES = [
    "Resource Use", "Emissions", "Innovation", 
    "Workforce", "Human Rights", "Community", "Product Responsibility", 
    "Management", "Shareholders", "CSR Strategy"
]

# Mapping from subcategory to parent category
PARENT_MAP = {
    "Resource Use": "E", "Emissions": "E", "Innovation": "E",
    "Workforce": "S", "Human Rights": "S", "Community": "S", "Product Responsibility": "S",
    "Management": "G", "Shareholders": "G", "CSR Strategy": "G",
    "None": "N"
}

API_KEY loaded succesfully


# Multistage classification algorithm (stages 1 & 2):

In [ ]:
client = genai.Client(api_key=API_KEY)

def load_definitions(filepath):
    """Loads ESG definitions from text file. Raises error if file is missing."""
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: Definition file '{filepath}' not found.")
        raise FileNotFoundError(f"Missing required file: {filepath}")
    
    with open(filepath, 'r', encoding='utf-8') as f:
        print(filepath, "opened successfully.")
        return f.read()

def load_examples(filepath):
    """Loads few-shot examples from text file. Raises error if file is missing."""
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: Examples file '{filepath}' not found.")
        raise FileNotFoundError(f"Missing required file: {filepath}")
        
    with open(filepath, 'r', encoding='utf-8') as f:
        print(filepath, "opened successfully.")
        return f.read()
    
def get_previous_output_string(row):
    """Reconstructs a readable JSON-like string from the dataframe row."""
    data = {}
    for cat in ["Cat_E", "Cat_S", "Cat_G", "Cat_N"]:
        data[cat] = row.get(cat, "No")
    
    for sub in SUBCATEGORIES:
        clean_sub = sub.replace(" ", "_")
        sub_dict = {"active": row.get(f"{clean_sub}_active", "No")}
        if REASONING_ENABLED:
            sub_dict["reasoning"] = row.get(f"{clean_sub}_reasoning", "")
        if CONFIDENCE_ENABLED:
            sub_dict["confidence"] = row.get(f"{clean_sub}_confidence", 0.0)
        data[clean_sub] = sub_dict
        
    return json.dumps(data, indent=2)

def build_prompt(post_text, definitions, examples, previous_results):
    """Dynamically builds the prompt based on configuration."""
    
    # Constructing the dynamic JSON schema description
    schema_fields = ['"Cat_E": "Yes/No"', '"Cat_S": "Yes/No"', '"Cat_G": "Yes/No"', '"Cat_N": "Yes/No"']
    for sub in SUBCATEGORIES:
        clean_sub = sub.replace(" ", "_")
        details = ['"active": "Yes/No"']
        if REASONING_ENABLED: details.append('"reasoning": "string"')
        if CONFIDENCE_ENABLED: details.append('"confidence": float(0-1)')
        schema_fields.append(f'"{clean_sub}": {{{", ".join(details)}}}')

    base_instructions = f"""
You are {ROLE}.
Your task is to analyze LinkedIn posts and classify them into 10 specific LSEG ESG subcategories.

Definitions of the LSEG ESG subcategories:
{definitions}

Instructions:
1. Determine if the article contains explicit thematic evidence of one or more of the 10 LSEG ESG categories. 
You are identifying the presence of a topic, regardless of whether the information is positive, negative, or neutral. 
Only classify based on explicit evidence. NEVER derive or imply categories.
2. If there is absolutely no concrete evidence for any category, assign "N" as the label and "None" as the sublabel. 
Even if you assign "None", you MUST {"provide a justification explaining why the article is not ESG-relevant and" if REASONING_ENABLED else ""} set all scores to 0. 
Never provide an empty output.
{ "3. Provide a detailed reasoning for every active subcategory classification." if REASONING_ENABLED else "" }
{ "4. Provide a confidence score (0.0 to 1.0) for every classification, representing your level of certainty." if CONFIDENCE_ENABLED else "" }

{ f"Examples:\n{examples}" if EXAMPLES_ENABLED else ""}

Strict Output Format (JSON):
{{
    {", ".join(schema_fields)}
}}

Input (LinkedIn Post):
{post_text}
"""
    if MULTISTAGE == 2:
            return f"""{base_instructions}
    ---
    INITIAL CLASSIFICATION (PHASE 1):
    {previous_results[0]}
    ---
    CRITICAL REVIEW TASK:
---
INITIAL ANALYSIS (PHASE 1):
{previous_results[0]}
---
TASK: OFFICIAL VERIFICATION
Your task is to produce the **Official Verified Classification** for the post below.

1. Review your Initial Analysis critically. Ensure you haven't implied categories, only explicit evidence counts.
2. If the Initial Analysis is correct, your "Official" output should match it exactly.
3. If you find an error, your "Official" output must reflect the correction.

STRICT RULE: You must output the FULL JSON structure for every post. Even if you agree with every single label from Phase 1, you must provide the complete JSON object as your final verified record.

Input (LinkedIn Post):
{post_text}
"""
    elif MULTISTAGE == 3:
        return f"""{base_instructions}
---
PHASE 1 OUTPUT: {previous_results[0]}
PHASE 2 OUTPUT: {previous_results[1]}
---
FINAL CONSENSUS TASK:
Compare the two previous classification attempts above. If they differ, decide which is more accurate according to LSEG standards. 
Provide the final, definitive classification.
TASK: FINAL CONSENSUS RECORD
Your task is to produce the **Final Definitive Classification** for the post below.

1. Review your Phase 1 and Phase 2 analyses critically. Ensure you haven't implied categories, only explicit evidence counts.
2. If Phase 1 and Phase 2 agree and are accurate, replicate that classification in your output.
3. If they disagree, or both aren't accurate, your "Final" output must reflect the correction.
4. Your output must be the complete JSON representation of the final decision.

STRICT RULE: Do not provide commentary. Provide only the full, valid JSON object representing the final consensus.

Input (LinkedIn Post):
{post_text}
"""

def get_response_schema():
    """Defines a strict JSON schema to physically prevent syntax errors like trailing commas."""
    properties = {
        "Cat_E": {"type": "STRING", "description": "Yes/No"},
        "Cat_S": {"type": "STRING", "description": "Yes/No"},
        "Cat_G": {"type": "STRING", "description": "Yes/No"},
        "Cat_N": {"type": "STRING", "description": "Yes/No"},
    }
    
    # Define the structure for each subcategory object
    sub_props = {
        "active": {"type": "STRING", "description": "Yes/No"}
    }
    if REASONING_ENABLED:
        sub_props["reasoning"] = {"type": "STRING"}
    if CONFIDENCE_ENABLED:
        sub_props["confidence"] = {"type": "NUMBER"}

    # Add each subcategory to the schema dynamically
    for sub in SUBCATEGORIES:
        clean_sub = sub.replace(" ", "_")
        properties[clean_sub] = {
            "type": "OBJECT",
            "properties": sub_props,
            "required": list(sub_props.keys())
        }
    
    return types.Schema(
        type="OBJECT",
        properties=properties,
        required=list(properties.keys())
    )

def classify_post(post_text, definitions, examples, retries, history):
    """Sends prompt to Gemini using a strict Response Schema and handles 503 errors."""
    prompt = build_prompt(post_text, definitions, examples, history)
    
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=TEMPERATURE,
                    response_mime_type="application/json",
                    response_schema=get_response_schema(), # This is the magic fix
                )
            )
            # The schema ensures the text is valid JSON
            return json.loads(response.text)
            
        except Exception as e:
            # Check for 503 (Service Unavailable) or 500 (Internal Error)
            if "503" in str(e) or "500" in str(e):
                print(f"\n[Attempt {attempt + 1}] Server busy (503/500). Retrying in {5 * (attempt + 1)}s...")
                time.sleep(5 * (attempt + 1))
                continue
            
            # For other errors (like auth or quota), return immediately
            return {"error": str(e)}
            
    return {"error": "Maximum retries reached. Google service is currently unavailable."}

def main():
    # 1. Verification of required files
    try:
        definitions = load_definitions(CATEGORY_FILE)
        examples = load_examples(EXAMPLES_FILE)
    except FileNotFoundError:
        sys.exit(1)

    if not os.path.exists(STAGE_ONE_FILE):
        print(f"Error: Input file '{STAGE_ONE_FILE}' not found.")
        return

    # 2. Load Data
    df = pd.read_csv(STAGE_ONE_FILE)
    print(STAGE_ONE_FILE, "opened succesfully")
    # Expected columns: Company, Date, Link, Text
    text_col = "Text" 
    # Load History Files based on STAGE
    df_ph1 = pd.read_csv(STAGE_ONE_FILE)
    if MULTISTAGE == 3: df_ph2 = pd.read_csv(STAGE_TWO_FILE)

    # 3. Initialize Columns
    # We explicitly set these to None so we don't keep old data from previous stages.
    
    # Clear Main Categories
    for cat in ["Cat_E", "Cat_S", "Cat_G", "Cat_N"]:
        df[cat] = None
        
    # Clear Subcategories (Option 1)
    for sub in SUBCATEGORIES:
        clean_sub = sub.replace(" ", "_")
        
        # Reset the 'active' status (Yes/No)
        df[f"{clean_sub}_active"] = None
        
        # Reset reasoning if enabled
        if REASONING_ENABLED:
            df[f"{clean_sub}_reasoning"] = None
            
        # Reset confidence if enabled
        if CONFIDENCE_ENABLED:
            df[f"{clean_sub}_confidence"] = None

    # Calculate range
    end_index = len(df)
    if MAX_POSTS is not None:
        end_index = min(POST_START + MAX_POSTS, len(df))

    print(f"Processing posts from index {POST_START} to {end_index}...")

    # 4. Processing Loop
    for index in tqdm(range(POST_START, end_index), desc="Classifying"):
        # Skip if already successfully processed
        if pd.notna(df.at[index, 'Cat_E']) and "ERROR" not in str(df.at[index, 'Cat_N']):
            continue

        post_content = df.at[index, text_col]

        # Gather History
        history = []
        # We always need Phase 1 results for Stage 2 or 3
        history.append(get_previous_output_string(df_ph1.iloc[index]))
        
        # If we are in Stage 3, we also need to have Phase 2 results available
        if MULTISTAGE == 3: history.append(get_previous_output_string(df_ph2.iloc[index]))

        result = classify_post(post_content, definitions, examples, RETRIES, history)

        if "error" in result:
            df.at[index, 'Cat_N'] = "ERROR: " + result["error"]
        else:
            # Map Main Categories
            for cat in ["Cat_E", "Cat_S", "Cat_G", "Cat_N"]:
                df.at[index, cat] = result.get(cat, "No")

            # Map Subcategories directly
            for sub in SUBCATEGORIES:
                clean_sub = sub.replace(" ", "_")
                sub_data = result.get(clean_sub, {})
                
                df.at[index, f"{clean_sub}_active"] = sub_data.get("active", "No")
                if REASONING_ENABLED:
                    df.at[index, f"{clean_sub}_reasoning"] = sub_data.get("reasoning", "")
                if CONFIDENCE_ENABLED:
                    df.at[index, f"{clean_sub}_confidence"] = sub_data.get("confidence", 0.0)

        # Checkpoint
        if (index + 1) % CHECKPOINT_INTERVAL == 0:
            df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
        
        time.sleep(SLEEP_TIME)

    # Final Save
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"\nProcessing complete! Results saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

classification_categories.txt opened successfully.
classification_examples.txt opened successfully.
PoC classification_Gem2.5f_role1_examples.csv opened succesfully
Processing posts from index 10 to 20...


Classifying: 100%|██████████| 10/10 [03:05<00:00, 18.53s/it]


Processing complete! Results saved to PoC classification_multi3_Gem2.5f_role1_examples.csv
